<a href="https://colab.research.google.com/github/AishahAbdulHakeem/skincare-ai/blob/main/01_data_cleaning_and_ingredient_analysis.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# 01 Data Cleaning and Ingredient Analysis

This notebook:
- Filters dataset to skincare
- Cleans ingredient lists
- Analyzes ingredient frequency distribution
- Selects statistically meaningful ingredient subset (≥50 occurrences)

In [ ]:
# --- Imports ---
import pandas as pd
import numpy as np
import ast
import re
import pickle
from collections import Counter

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [ ]:
#Have to create skincare-ai/data/raw path and upload csv to that path
PRODUCT_PATH = "/content/drive/MyDrive/skincare-ai/data/raw/product_info.csv"

products = pd.read_csv(PRODUCT_PATH)
print("Total products:", len(products))
products.head()

Total products: 8494


,product_id,product_name,brand_id,brand_name,loves_count,rating,reviews,size,variation_type,variation_value,...,online_only,out_of_stock,sephora_exclusive,highlights,primary_category,secondary_category,tertiary_category,child_count,child_max_price,child_min_price
0,P473671,Fragrance Discovery Set,6342,19-69,6320,3.6364,11.0,NaN,NaN,NaN,...,1,0,0,"['Unisex/ Genderless Scent', 'Warm &Spicy Scen...",Fragrance,Value & Gift Sets,Perfume Gift Sets,0,NaN,NaN
1,P473668,La Habana Eau de Parfum,6342,19-69,3827,4.1538,13.0,3.4 oz/ 100 mL,Size + Concentration + Formulation,3.4 oz/ 100 mL,...,1,0,0,"['Unisex/ Genderless Scent', 'Layerable Scent'...",Fragrance,Women,Perfume,2,85.0,30.0
2,P473662,Rainbow Bar Eau de Parfum,6342,19-69,3253,4.2500,16.0,3.4 oz/ 100 mL,Size + Concentration + Formulation,3.4 oz/ 100 mL,...,1,0,0,"['Unisex/ Genderless Scent', 'Layerable Scent'...",Fragrance,Women,Perfume,2,75.0,30.0
3,P473660,Kasbah Eau de Parfum,6342,19-69,3018,4.4762,21.0,3.4 oz/ 100 mL,Size + Concentration + Formulation,3.4 oz/ 100 mL,...,1,0,0,"['Unisex/ Genderless Scent', 'Layerable Scent'...",Fragrance,Women,Perfume,2,75.0,30.0
4,P473658,Purple Haze Eau de Parfum,6342,19-69,2691,3.2308,13.0,3.4 oz/ 100 mL,Size + Concentration + Formulation,3.4 oz/ 100 mL,...,1,0,0,"['Unisex/ Genderless Scent', 'Layerable Scent'...",Fragrance,Women,Perfume,2,75.0,30.0


# Filter Skin Care Products

In [ ]:
skincare = products[products['primary_category'] == 'Skincare'].copy()

print("Total products:", len(products))
print("Skincare products:", len(skincare))

Total products: 8494
Skincare products: 2420


In [ ]:
skincare['ingredients'].head()

,ingredients
89,"['Collagen (Vegan)*, Water (Aqua, Eau), Ethylh..."
90,"['Collagen (Vegan)*, Water (Aqua, Eau), Propan..."
91,"['Aqua (Water/Eau), Stearic Acid, Isopropyl Is..."
92,"['Collagen (Vegan)*, Water (Aqua, Eau), Glycer..."
93,"['Octinoxate 7.5%, Titanium Dioxide 2%, Zinc O..."


In [ ]:
def clean_ingredients(ingredient_string):
    if pd.isna(ingredient_string):
        return []

    try:
        # Convert string representation to actual Python list
        parsed = ast.literal_eval(ingredient_string)

        if isinstance(parsed, list) and len(parsed) > 0:
            ingredient_text = parsed[0]
        else:
            return []

        # Split on commas NOT inside parentheses
        ingredients = re.split(r',(?![^()]*\))', ingredient_text)

        # Clean formatting
        ingredients = [ing.strip().lower() for ing in ingredients if ing.strip() != ""]

        return ingredients

    except:
        return []

skincare['ingredient_list'] = skincare['ingredients'].apply(clean_ingredients)


<unknown>:1: SyntaxWarning: invalid escape sequence '\A'
<unknown>:1: SyntaxWarning: invalid escape sequence '\A'


In [ ]:
skincare['ingredient_list'] = skincare['ingredients'].apply(clean_ingredients)

<unknown>:1: SyntaxWarning: invalid escape sequence '\A'
<unknown>:1: SyntaxWarning: invalid escape sequence '\A'


In [ ]:
# Standardize water variants and remove parsing errors

water_variants = {
    'water/aqua/eau', 'water\\aqua\\eau', 'aqua',
    'aqua/water', 'eau', 'aqua/water/eau',
    'water (aqua)', 'purified water', 'deionized water'
}

# Known junk from parsing errors
junk = {'1', '2', '3', '4', '5', ''}

def normalize_ingredients(ing_list):
    normalized = []
    for ing in ing_list:
        if ing in junk:
            continue              # drop parsing errors entirely
        elif ing in water_variants:
            normalized.append('water')  # standardize all water variants
        else:
            normalized.append(ing)
    return normalized

skincare['ingredient_list'] = skincare['ingredient_list'].apply(normalize_ingredients)

# Verify fixes
print("Water variants remaining:",
      sum(1 for ings in skincare['ingredient_list']
          for i in ings if i in water_variants))

print("Junk entries remaining:",
      sum(1 for ings in skincare['ingredient_list']
          for i in ings if i in junk))

Water variants remaining: 0
Junk entries remaining: 0


In [ ]:
skincare[['ingredients', 'ingredient_list']].head(3)

,ingredients,ingredient_list
89,"['Collagen (Vegan)*, Water (Aqua, Eau), Ethylh...","[collagen (vegan)*, water (aqua, eau), ethylhe..."
90,"['Collagen (Vegan)*, Water (Aqua, Eau), Propan...","[collagen (vegan)*, water (aqua, eau), propane..."
91,"['Aqua (Water/Eau), Stearic Acid, Isopropyl Is...","[aqua (water/eau), stearic acid, isopropyl iso..."


# Frequency Analysis

In [ ]:
all_ingredients = skincare['ingredient_list'].explode()

ingredient_counts = Counter(all_ingredients)

ingredient_freq = pd.DataFrame(
    ingredient_counts.items(),
    columns=['ingredient', 'count']
).sort_values(by='count', ascending=False).reset_index(drop=True)

ingredient_freq.head(20)

,ingredient,count
0,glycerin,1542
1,water,1371
2,phenoxyethanol,930
3,butylene glycol,923
4,sodium hyaluronate,729
5,tocopherol,724
6,citric acid,722
7,propanediol,706
8,xanthan gum,669
9,ethylhexylglycerin,647


In [ ]:
print("Total unique ingredients:", ingredient_freq.shape[0])

print("Ingredients appearing ≥ 10 times:",
      ingredient_freq[ingredient_freq['count'] >= 10].shape[0])

print("Ingredients appearing ≥ 25 times:",
      ingredient_freq[ingredient_freq['count'] >= 25].shape[0])

print("Ingredients appearing ≥ 50 times:",
      ingredient_freq[ingredient_freq['count'] >= 50].shape[0])

Total unique ingredients: 7254
Ingredients appearing ≥ 10 times: 1039
Ingredients appearing ≥ 25 times: 467
Ingredients appearing ≥ 50 times: 219


# Top Ingredient Selection

In [ ]:
top_ingredients = set(
    ingredient_freq[ingredient_freq['count'] >= 50]['ingredient']
)

print("Top ingredients selected:", len(top_ingredients))

Top ingredients selected: 219


# Saving Cleaned Dataset

In [ ]:
import os
os.makedirs("/content/drive/MyDrive/skincare-ai/data/processed", exist_ok=True)

processed_skincare = skincare[['product_id', 'rating', 'ingredient_list']].copy()

processed_skincare.to_pickle(
    "/content/drive/MyDrive/skincare-ai/data/processed/skincare_cleaned.pkl"
)

with open("/content/drive/MyDrive/skincare-ai/data/processed/top_ingredients.pkl", "wb") as f:
    pickle.dump(top_ingredients, f)

print("Saved: skincare_cleaned.pkl")
print("Saved: top_ingredients.pkl")
print("Products saved:", len(processed_skincare))
print("Top ingredients saved:", len(top_ingredients))

Saved: skincare_cleaned.pkl
Saved: top_ingredients.pkl
Products saved: 2420
Top ingredients saved: 219
